In [2]:
# Étape 30 — Charger le référentiel dans le notebook des comptes
# Je charge le référentiel pour retrouver le CIK de chaque action.
# Je lis les CIK comme du texte pour conserver leurs zéros initiaux.
import pandas as pd

titres = pd.read_csv("data/processed/referentiel_titres.csv",dtype={"CIK": str},)

display(titres[["Ticker", "CIK"]])

,Ticker,CIK
0,AMD,0000002488
1,NVDA,0001045810
2,INTC,0000050863
3,AVGO,0001730168
4,MRVL,0001835632


In [3]:
# Étape 31 — Télécharger les données comptables des cinq entreprises
# Je télécharge les données comptables de chaque entreprise avec son CIK.
# Je conserve les réponses originales de la SEC dans data/raw/sec.

from getpass import getpass
from pathlib import Path
import time

import requests

contact_sec = getpass("Adresse e-mail de contact pour la SEC : ").strip()
dossier_sec = Path("data/raw/sec")
dossier_sec.mkdir(parents=True, exist_ok=True)

for ticker, cik in titres[["Ticker", "CIK"]].itertuples(index=False, name=None):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    reponse = requests.get(
        url,
        headers={"User-Agent": f"Select Equity {contact_sec}"},
        timeout=30,
    )
    reponse.raise_for_status()

    donnees_sec = reponse.json()
    if int(donnees_sec["cik"]) != int(cik):
        raise ValueError(f"Le CIK renvoyé par la SEC ne correspond pas à {ticker}.")

    fichier = dossier_sec / f"{ticker}_companyfacts.json"
    fichier.write_bytes(reponse.content)
    print(f"{ticker} — {donnees_sec['entityName']} — {fichier.name}")
    time.sleep(0.5)

Adresse e-mail de contact pour la SEC :  ········


AMD — ADVANCED MICRO DEVICES, INC — AMD_companyfacts.json
NVDA — NVIDIA CORP — NVDA_companyfacts.json
INTC — Intel Corporation — INTC_companyfacts.json
AVGO — Broadcom Inc. — AVGO_companyfacts.json
MRVL — MARVELL TECHNOLOGY, INC — MRVL_companyfacts.json


In [4]:
# Étape 32 — Sélectionner les bilans disponibles à la date d’analyse
# Je repère les dates de bilan à partir de la ligne de trésorerie.
# Je retiens le bilan le plus récent déjà publié au 31/12/2025.

import json

date_analyse = "2025-12-31"
bilans_retenus = []

for ticker in titres["Ticker"]:
    fichier = dossier_sec / f"{ticker}_companyfacts.json"
    donnees = json.loads(fichier.read_text(encoding="utf-8"))
    tresorerie = donnees["facts"]["us-gaap"][
        "CashAndCashEquivalentsAtCarryingValue"
    ]["units"]["USD"]

    candidats = [
        ligne for ligne in tresorerie
        if ligne["form"] in ("10-K", "10-Q")
        and ligne["filed"] <= date_analyse
        and ligne["end"] <= date_analyse
    ]

    if not candidats:
        raise ValueError(f"Aucun bilan disponible pour {ticker} au {date_analyse}.")

    bilan = max(candidats, key=lambda ligne: (ligne["end"], ligne["filed"]))
    bilans_retenus.append({
        "Ticker": ticker,
        "Rapport": bilan["form"],
        "Date_bilan": bilan["end"],
        "Date_depot": bilan["filed"],
        "Numero_depot": bilan["accn"],
    })

bilans = pd.DataFrame(bilans_retenus)
display(bilans)


,Ticker,Rapport,Date_bilan,Date_depot,Numero_depot
0,AMD,10-Q,2025-09-27,2025-11-05,0000002488-25-000166
1,NVDA,10-Q,2025-10-26,2025-11-19,0001045810-25-000230
2,INTC,10-Q,2025-09-27,2025-11-06,0000050863-25-000179
3,AVGO,10-K,2025-11-02,2025-12-18,0001730168-25-000121
4,MRVL,10-Q,2025-11-01,2025-12-03,0001835632-25-000197


In [5]:
# Étape 33 — Extraire la trésorerie et les dettes
# Je récupère la trésorerie et les deux postes de dette du bilan retenu.
# Je laisse les montants à vérifier vides et j'affiche les chiffres en millions USD.

champs_bilan = {
    "Tresorerie_MUSD": "CashAndCashEquivalentsAtCarryingValue",
    "Dette_courante_MUSD": "LongTermDebtCurrent",
    "Dette_non_courante_MUSD": "LongTermDebtNoncurrent",
}

lignes_bilan = []

for bilan in bilans.to_dict("records"):
    fichier = dossier_sec / f"{bilan['Ticker']}_companyfacts.json"
    donnees = json.loads(fichier.read_text(encoding="utf-8"))
    faits = donnees["facts"]["us-gaap"]
    resultat = bilan.copy()

    for colonne, champ in champs_bilan.items():
        observations = faits.get(champ, {}).get("units", {}).get("USD", [])
        montants = {
            ligne["val"]
            for ligne in observations
            if ligne["accn"] == bilan["Numero_depot"]
            and ligne["end"] == bilan["Date_bilan"]
        }
        resultat[colonne] = montants.pop() / 1_000_000 if len(montants) == 1 else None

    lignes_bilan.append(resultat)

postes_bilan = pd.DataFrame(lignes_bilan)
display(postes_bilan[["Ticker", *champs_bilan]])

,Ticker,Tresorerie_MUSD,Dette_courante_MUSD,Dette_non_courante_MUSD
0,AMD,4808.0,873.0,2347.0
1,NVDA,11486.0,999.0,7468.0
2,INTC,11141.0,NaN,44057.0
3,AVGO,16178.0,3152.0,NaN
4,MRVL,2714.5,NaN,3969.4


In [6]:
# Étape 34 — Repérer les champs SEC qui correspondent à ces montants
# Je cherche, dans le dépôt et à la date du bilan retenus,
# les champs de dette qui donnent le montant vérifié dans le rapport.

montants_controles_usd = {
    "INTC": 2_496_000_000,
    "AVGO": 61_984_000_000,
    "MRVL": 499_500_000,
}

bilans_par_ticker = bilans.set_index("Ticker")

for ticker, montant in montants_controles_usd.items():
    bilan = bilans_par_ticker.loc[ticker]
    fichier = dossier_sec / f"{ticker}_companyfacts.json"
    donnees = json.loads(fichier.read_text(encoding="utf-8"))
    champs_trouves = []

    for champ, detail in donnees["facts"]["us-gaap"].items():
        if "debt" not in champ.lower() and "borrow" not in champ.lower():
            continue

        if any(
            ligne["accn"] == bilan["Numero_depot"]
            and ligne["end"] == bilan["Date_bilan"]
            and "start" not in ligne
            and ligne["val"] == montant
            for ligne in detail.get("units", {}).get("USD", [])
        ):
            champs_trouves.append(champ)

    print(ticker, champs_trouves)
    

INTC ['DebtCurrent']
AVGO ['LongTermDebtAndCapitalLeaseObligations']
MRVL ['ShortTermBorrowings']


In [10]:
# Étape 35 — Adapter l’extraction aux trois entreprises
# Je lis le bilan retenu pour chaque entreprise.
# J'utilise les champs SEC vérifiés pour récupérer la trésorerie et les dettes.

champs_bilan = {
    "Tresorerie_MUSD": "CashAndCashEquivalentsAtCarryingValue",
    "Dette_courante_MUSD": "LongTermDebtCurrent",
    "Dette_non_courante_MUSD": "LongTermDebtNoncurrent",
}

champs_specifiques = {
    ("INTC", "Dette_courante_MUSD"): "DebtCurrent",
    ("AVGO", "Dette_non_courante_MUSD"): "LongTermDebtAndCapitalLeaseObligations",
    ("MRVL", "Dette_courante_MUSD"): "ShortTermBorrowings",
}

lignes_bilan = []

for bilan in bilans.to_dict("records"):
    ticker = bilan["Ticker"]
    fichier = dossier_sec / f"{ticker}_companyfacts.json"
    donnees = json.loads(fichier.read_text(encoding="utf-8"))
    faits = donnees["facts"]["us-gaap"]
    resultat = bilan.copy()

    for colonne, champ_standard in champs_bilan.items():
        champ = champs_specifiques.get((ticker, colonne), champ_standard)
        observations = faits.get(champ, {}).get("units", {}).get("USD", [])
        montants = {
            ligne["val"]
            for ligne in observations
            if ligne["accn"] == bilan["Numero_depot"]
            and ligne["end"] == bilan["Date_bilan"]
            and "start" not in ligne
        }
        resultat[colonne] = montants.pop() / 1_000_000 if len(montants) == 1 else None

    lignes_bilan.append(resultat)

postes_bilan = pd.DataFrame(lignes_bilan)
display(postes_bilan[["Ticker", *champs_bilan]])

,Ticker,Tresorerie_MUSD,Dette_courante_MUSD,Dette_non_courante_MUSD
0,AMD,4808.0,873.0,2347.0
1,NVDA,11486.0,999.0,7468.0
2,INTC,11141.0,2496.0,44057.0
3,AVGO,16178.0,3152.0,61984.0
4,MRVL,2714.5,499.5,3969.4


In [11]:
# Étape 36 — Calculer la dette nette des cinq entreprises
# Je calcule la dette nette en millions USD à partir du bilan retenu.
# Je conserve les dates du bilan et du dépôt pour pouvoir vérifier le résultat.

postes_bilan["Dette_nette_MUSD"] = (
    postes_bilan["Dette_courante_MUSD"]
    + postes_bilan["Dette_non_courante_MUSD"]
    - postes_bilan["Tresorerie_MUSD"]
)

display(postes_bilan[["Ticker", "Date_bilan", "Date_depot", "Dette_nette_MUSD"]])

,Ticker,Date_bilan,Date_depot,Dette_nette_MUSD
0,AMD,2025-09-27,2025-11-05,-1588.0
1,NVDA,2025-10-26,2025-11-19,-3019.0
2,INTC,2025-09-27,2025-11-06,35412.0
3,AVGO,2025-11-02,2025-12-18,48958.0
4,MRVL,2025-11-01,2025-12-03,1754.4


In [12]:
# Étape 37 — Calculer la marge opérationnelle trimestrielle
# Je repère le trimestre dans le rapport retenu.
# Je calcule la marge avec les ventes et le résultat de la même période.

def montants_du_rapport(faits, champ, bilan):
    observations = faits.get(champ, {}).get("units", {}).get("USD", [])
    return [
        ligne for ligne in observations
        if ligne["accn"] == bilan["Numero_depot"]
        and ligne["end"] == bilan["Date_bilan"]
        and ligne.get("start")
    ]


lignes_marge = []

for bilan in bilans.to_dict("records"):
    ticker = bilan["Ticker"]
    fichier = dossier_sec / f"{ticker}_companyfacts.json"
    faits = json.loads(fichier.read_text(encoding="utf-8"))["facts"]["us-gaap"]

    resultats = montants_du_rapport(faits, "OperatingIncomeLoss", bilan)
    trimestres = {
        (ligne["start"], ligne["val"])
        for ligne in resultats
        if 75 <= (pd.Timestamp(ligne["end"]) - pd.Timestamp(ligne["start"])).days <= 105
    }

    ligne_marge = {
        "Ticker": ticker,
        "Debut_trimestre": None,
        "Marge_operationnelle_pct": None,
    }

    if len(trimestres) == 1:
        debut, resultat_operationnel = trimestres.pop()
        ligne_marge["Debut_trimestre"] = debut

        for champ in ("RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues"):
            ventes = {
                ligne["val"]
                for ligne in montants_du_rapport(faits, champ, bilan)
                if ligne["start"] == debut
            }
            if len(ventes) == 1:
                chiffre_affaires = ventes.pop()
                if chiffre_affaires > 0:
                    ligne_marge["Marge_operationnelle_pct"] = (
                        resultat_operationnel / chiffre_affaires * 100
                    )
                break

    lignes_marge.append(ligne_marge)

marges = pd.DataFrame(lignes_marge)
display(marges.round(2))

,Ticker,Debut_trimestre,Marge_operationnelle_pct
0,AMD,2025-06-29,13.74
1,NVDA,2025-07-28,63.17
2,INTC,2025-06-29,5.00
3,AVGO,None,NaN
4,MRVL,2025-08-03,17.25


In [13]:
# Étape 38 — Compléter la marge de Broadcom et tracer sa source
# Je complète le T4 de Broadcom avec les chiffres GAAP du communiqué
# joint au 8-K du 11/12/2025, déjà publié à la date d'analyse.

avgo = marges["Ticker"].eq("AVGO")
ventes_avgo_musd = 18_015
resultat_operationnel_avgo_musd = 7_508

marges.loc[avgo, "Marge_operationnelle_pct"] = (
    resultat_operationnel_avgo_musd / ventes_avgo_musd * 100
)
marges.loc[avgo, "Source_marge"] = (
    "https://www.sec.gov/Archives/edgar/data/1730168/"
    "000173016825000116/avgo-11022025x8kxex99.htm"
)

display(marges.loc[avgo, ["Ticker", "Marge_operationnelle_pct"]].round(2))

,Ticker,Marge_operationnelle_pct
3,AVGO,41.68


In [14]:
# Étape 39 — Vérifier les BPA disponibles pour le P/E
# Je liste les BPA dilués trimestriels publiés au plus tard le 31/12/2025.
# J'affiche les dates pour repérer les trimestres manquants avant de calculer le P/E.

lignes_bpa = []

for ticker in titres["Ticker"]:
    fichier = dossier_sec / f"{ticker}_companyfacts.json"
    faits = json.loads(fichier.read_text(encoding="utf-8"))["facts"]["us-gaap"]
    observations = faits.get("EarningsPerShareDiluted", {}).get("units", {}).get(
        "USD/shares", []
    )

    for ligne in observations:
        debut = ligne.get("start")
        fin = ligne["end"]

        if not debut or not ("2024-10-01" <= fin <= date_analyse):
            continue
        if ligne["filed"] > date_analyse or ligne["form"] not in ("10-K", "10-Q"):
            continue

        duree = (pd.Timestamp(fin) - pd.Timestamp(debut)).days
        if 75 <= duree <= 105:
            lignes_bpa.append({
                "Ticker": ticker,
                "Debut_trimestre": debut,
                "Fin_trimestre": fin,
                "Date_depot": ligne["filed"],
                "BPA_dilue_USD": ligne["val"],
            })

bpa_disponibles = (
    pd.DataFrame(lignes_bpa)
    .drop_duplicates()
    .sort_values(["Ticker", "Fin_trimestre", "Date_depot"])
)

display(bpa_disponibles)

,Ticker,Debut_trimestre,Fin_trimestre,Date_depot,BPA_dilue_USD
0,AMD,2024-12-29,2025-03-29,2025-05-07,0.44
1,AMD,2025-03-30,2025-06-28,2025-08-06,0.54
2,AMD,2025-06-29,2025-09-27,2025-11-05,0.75
11,AVGO,2024-11-04,2025-02-02,2025-03-12,1.14
12,AVGO,2025-02-03,2025-05-04,2025-06-11,1.03
13,AVGO,2025-05-05,2025-08-03,2025-09-10,0.85
8,INTC,2024-12-29,2025-03-29,2025-04-25,-0.19
9,INTC,2025-03-30,2025-06-28,2025-07-24,-0.67
10,INTC,2025-06-29,2025-09-27,2025-11-06,0.90
14,MRVL,2024-08-04,2024-11-02,2024-12-04,-0.78


Le tableau montre qu’il manque un trimestre par entreprise. NVIDIA et Marvell ont déjà quatre dates visibles, mais il y a un trou entre les trimestres de fin 2024 et ceux de 2025. Pour le P/E, il faut quatre trimestres consécutifs. Un chiffre repris dans deux rapports compte une seule fois.

In [15]:
# Étape 40 — Compléter les BPA des trimestres manquants
# J'ajoute les BPA vérifiés dans les communiqués officiels.
# Je conserve une seule ligne par trimestre et je vérifie la continuité des dates.

complements_bpa = pd.DataFrame(
    [
        ("AMD",  "2024-09-29", "2024-12-28", "2025-02-04",  0.29),
        ("NVDA", "2024-10-28", "2025-01-26", "2025-02-26",  0.89),
        ("INTC", "2024-09-29", "2024-12-28", "2025-01-30", -0.03),
        ("AVGO", "2025-08-04", "2025-11-02", "2025-12-11",  1.74),
        ("MRVL", "2024-11-03", "2025-02-01", "2025-03-05",  0.23),
    ],
    columns=["Ticker", "Debut_trimestre", "Fin_trimestre",
             "Date_publication", "BPA_dilue_USD"],
)

bpa_sec = bpa_disponibles.rename(columns={"Date_depot": "Date_publication"})
bpa_complets = pd.concat(
    [bpa_sec.assign(Source="SEC Company Facts"),
     complements_bpa.assign(Source="Communiqué officiel")],
    ignore_index=True,
)

bpa_4_trimestres = (
    bpa_complets
    .sort_values(["Ticker", "Fin_trimestre", "Date_publication"])
    .drop_duplicates(["Ticker", "Fin_trimestre"], keep="last")
    .groupby("Ticker")
    .tail(4)
)

for ticker, groupe in bpa_4_trimestres.groupby("Ticker"):
    debuts = pd.to_datetime(groupe["Debut_trimestre"])
    fins = pd.to_datetime(groupe["Fin_trimestre"])
    ecarts = debuts - fins.shift()
    if len(groupe) != 4 or not ecarts.iloc[1:].eq(pd.Timedelta(days=1)).all():
        raise ValueError(f"Les quatre trimestres de {ticker} sont à vérifier.")

controle_bpa = bpa_4_trimestres.groupby("Ticker").agg(
    Nombre_trimestres=("Fin_trimestre", "count"),
    BPA_4_trimestres_USD=("BPA_dilue_USD", "sum"),
)

display(controle_bpa.round(2))

,Nombre_trimestres,BPA_4_trimestres_USD
Ticker,,
AMD,4,2.02
AVGO,4,4.76
INTC,4,0.01
MRVL,4,2.85
NVDA,4,4.03


In [16]:
# Étape 41 — Calculer le P/E des cinq actions
# Je récupère la clôture de chaque action à la date d'analyse.
# Je la divise par le BPA cumulé si celui-ci est positif.

cours_journaliers = pd.read_csv(
    "data/processed/cours_journaliers_2021_2025.csv"
)

cours_analyse = cours_journaliers.loc[
    cours_journaliers["Date"].eq(date_analyse),
    ["Ticker", "Cloture"],
]

valorisation = controle_bpa.reset_index().merge(
    cours_analyse, on="Ticker", how="left", validate="one_to_one"
)

bpa = valorisation["BPA_4_trimestres_USD"]
valorisation["PE_historique"] = valorisation["Cloture"] / bpa.where(bpa > 0)

display(valorisation[[
    "Ticker", "Cloture", "BPA_4_trimestres_USD", "PE_historique"
]].round(2))

,Ticker,Cloture,BPA_4_trimestres_USD,PE_historique
0,AMD,214.16,2.02,106.02
1,AVGO,346.10,4.76,72.71
2,INTC,36.90,0.01,3690.00
3,MRVL,84.98,2.85,29.82
4,NVDA,186.50,4.03,46.28


Les cinq P/E s’affichent, et AMD retrouve bien 106,02×. Pour Intel, 3 690× est le résultat du calcul avec le BPA cumulé de 0,01 USD. Comme ce BPA est proche de zéro, le multiple est très sensible à l’arrondi. 

In [17]:
# Étape 42 — Regrouper les indicateurs financiers dans un CSV
# Je réunis les trois indicateurs financiers par ticker.
# Je garde les dates des comptes et j'enregistre la synthèse pour la suite du projet.

synthese_financiere = (
    postes_bilan[["Ticker", "Date_bilan", "Date_depot", "Dette_nette_MUSD"]]
    .merge(
        marges[["Ticker", "Marge_operationnelle_pct"]],
        on="Ticker",
        validate="one_to_one",
    )
    .merge(
        valorisation[["Ticker", "BPA_4_trimestres_USD", "PE_historique"]],
        on="Ticker",
        validate="one_to_one",
    )
)

synthese_financiere["Date_analyse"] = date_analyse

fichier_financier = "data/processed/synthese_financiere_2025_12_31.csv"
synthese_financiere.to_csv(fichier_financier, index=False)

controle_financier = pd.read_csv(fichier_financier)
print("Lignes relues :", len(controle_financier))
display(controle_financier.round(2))

Lignes relues : 5


,Ticker,Date_bilan,Date_depot,Dette_nette_MUSD,Marge_operationnelle_pct,BPA_4_trimestres_USD,PE_historique,Date_analyse
0,AMD,2025-09-27,2025-11-05,-1588.0,13.74,2.02,106.02,2025-12-31
1,NVDA,2025-10-26,2025-11-19,-3019.0,63.17,4.03,46.28,2025-12-31
2,INTC,2025-09-27,2025-11-06,35412.0,5.00,0.01,3690.00,2025-12-31
3,AVGO,2025-11-02,2025-12-18,48958.0,41.68,4.76,72.71,2025-12-31
4,MRVL,2025-11-01,2025-12-03,1754.4,17.25,2.85,29.82,2025-12-31


validate="one_to_one" vérifie que chaque ticker apparaît une seule fois de chaque côté de la jointure.